# RPS Experiment Results Analysis

Compare DCUNet with and without RPS conditioning on DREGON-LibriMix dataset.

Metrics:
- SI-SDR (Scale-Invariant Signal-to-Distortion Ratio)
- ESTOI (Extended Short-Time Objective Intelligibility)
- PESQ (Perceptual Evaluation of Speech Quality)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
from tqdm import tqdm

import soundfile as sf
import librosa
import librosa.display
import IPython.display as ipd

# Metrics
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio
from pesq import pesq
from pystoi import stoi
import torch

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

In [ ]:
# Configuration
RESULTS_RPS = Path("../results/dcunet_rps_dregon")
RESULTS_BASELINE = Path("../results/dcunet_baseline_dregon")
DATASET_DIR = Path("../datasets/DREGON-LM")
SAMPLE_RATE = 16000

# Check paths exist
print(f"RPS results: {RESULTS_RPS.exists()}")
print(f"Baseline results: {RESULTS_BASELINE.exists()}")
print(f"Dataset: {DATASET_DIR.exists()}")

## Load Training Logs

In [ ]:
def load_training_log(results_dir):
    """Load training metrics from wandb or log files."""
    # Try to find wandb run directory
    wandb_dirs = list(Path(results_dir).parent.glob("wandb/run-*"))
    
    # Try to load from a summary file if available
    log_file = results_dir / "training_log.json"
    if log_file.exists():
        with open(log_file) as f:
            return json.load(f)
    
    return None

rps_log = load_training_log(RESULTS_RPS)
baseline_log = load_training_log(RESULTS_BASELINE)

## Compute Evaluation Metrics

In [ ]:
def compute_metrics(clean_path, enhanced_path, mixture_path=None):
    """Compute SI-SDR, ESTOI, and PESQ for a single sample."""
    # Load audio
    clean, sr = sf.read(clean_path)
    enhanced, _ = sf.read(enhanced_path)
    
    # Ensure same length
    min_len = min(len(clean), len(enhanced))
    clean = clean[:min_len]
    enhanced = enhanced[:min_len]
    
    # SI-SDR
    si_sdr = ScaleInvariantSignalDistortionRatio()
    si_sdr_value = si_sdr(
        torch.tensor(enhanced).unsqueeze(0),
        torch.tensor(clean).unsqueeze(0)
    ).item()
    
    # ESTOI
    estoi_value = stoi(clean, enhanced, sr, extended=True)
    
    # PESQ
    try:
        pesq_value = pesq(sr, clean, enhanced, 'wb')  # Wideband
    except Exception:
        pesq_value = np.nan
    
    result = {
        'si_sdr': si_sdr_value,
        'estoi': estoi_value,
        'pesq': pesq_value,
    }
    
    # Also compute input metrics if mixture is provided
    if mixture_path is not None and Path(mixture_path).exists():
        mixture, _ = sf.read(mixture_path)
        mixture = mixture[:min_len]
        
        result['input_si_sdr'] = si_sdr(
            torch.tensor(mixture).unsqueeze(0),
            torch.tensor(clean).unsqueeze(0)
        ).item()
        result['input_estoi'] = stoi(clean, mixture, sr, extended=True)
        try:
            result['input_pesq'] = pesq(sr, clean, mixture, 'wb')
        except Exception:
            result['input_pesq'] = np.nan
    
    return result

In [ ]:
def evaluate_model(results_dir, dataset_dir, split='valid', max_samples=None):
    """Evaluate model outputs against ground truth."""
    eval_dir = results_dir / "eval_output"
    if not eval_dir.exists():
        print(f"Evaluation output not found at {eval_dir}")
        return None
    
    # Find all enhanced samples
    enhanced_files = sorted(glob(str(eval_dir / "**/vocals.wav"), recursive=True))
    if max_samples:
        enhanced_files = enhanced_files[:max_samples]
    
    results = []
    for enhanced_path in tqdm(enhanced_files, desc="Computing metrics"):
        # Get sample name
        sample_name = Path(enhanced_path).parent.name
        
        # Find corresponding clean and mixture files
        clean_path = dataset_dir / split / sample_name / "vocals.wav"
        mixture_path = dataset_dir / split / sample_name / "mixture.wav"
        
        if not clean_path.exists():
            continue
        
        metrics = compute_metrics(clean_path, enhanced_path, mixture_path)
        metrics['sample'] = sample_name
        results.append(metrics)
    
    return pd.DataFrame(results)

In [ ]:
# Evaluate both models
print("Evaluating RPS model...")
rps_metrics = evaluate_model(RESULTS_RPS, DATASET_DIR)

print("\nEvaluating baseline model...")
baseline_metrics = evaluate_model(RESULTS_BASELINE, DATASET_DIR)

## Results Summary

In [ ]:
def summarize_metrics(df, name):
    """Print summary statistics for metrics."""
    if df is None or len(df) == 0:
        print(f"{name}: No data available")
        return
    
    print(f"\n{name} ({len(df)} samples)")
    print("=" * 50)
    
    for metric in ['si_sdr', 'estoi', 'pesq']:
        if metric in df.columns:
            values = df[metric].dropna()
            print(f"{metric.upper():>10}: {values.mean():.4f} +/- {values.std():.4f}")
    
    # Improvement over input
    if 'input_si_sdr' in df.columns:
        improvement = df['si_sdr'] - df['input_si_sdr']
        print(f"\nSI-SDR improvement: {improvement.mean():.4f} +/- {improvement.std():.4f} dB")

summarize_metrics(rps_metrics, "DCUNet + RPS")
summarize_metrics(baseline_metrics, "DCUNet Baseline")

In [ ]:
# Create comparison DataFrame
if rps_metrics is not None and baseline_metrics is not None:
    comparison = pd.DataFrame({
        'Metric': ['SI-SDR (dB)', 'ESTOI', 'PESQ'],
        'DCUNet + RPS': [
            f"{rps_metrics['si_sdr'].mean():.2f} +/- {rps_metrics['si_sdr'].std():.2f}",
            f"{rps_metrics['estoi'].mean():.3f} +/- {rps_metrics['estoi'].std():.3f}",
            f"{rps_metrics['pesq'].mean():.2f} +/- {rps_metrics['pesq'].std():.2f}",
        ],
        'DCUNet Baseline': [
            f"{baseline_metrics['si_sdr'].mean():.2f} +/- {baseline_metrics['si_sdr'].std():.2f}",
            f"{baseline_metrics['estoi'].mean():.3f} +/- {baseline_metrics['estoi'].std():.3f}",
            f"{baseline_metrics['pesq'].mean():.2f} +/- {baseline_metrics['pesq'].std():.2f}",
        ],
    })
    display(comparison)

## Visualization

In [ ]:
def plot_metric_comparison(rps_df, baseline_df, metric, title):
    """Plot box plot comparison for a metric."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    data = []
    labels = []
    
    if rps_df is not None and metric in rps_df.columns:
        data.append(rps_df[metric].dropna().values)
        labels.append('DCUNet + RPS')
    
    if baseline_df is not None and metric in baseline_df.columns:
        data.append(baseline_df[metric].dropna().values)
        labels.append('DCUNet Baseline')
    
    if len(data) > 0:
        bp = ax.boxplot(data, labels=labels, patch_artist=True)
        colors = ['#66c2a5', '#fc8d62']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_ylabel(title)
        ax.set_title(f'{title} Comparison')
        
        # Add mean markers
        for i, d in enumerate(data):
            ax.scatter(i + 1, np.mean(d), marker='D', color='red', s=50, zorder=3)
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot all metrics
fig_sisdr = plot_metric_comparison(rps_metrics, baseline_metrics, 'si_sdr', 'SI-SDR (dB)')
plt.show()

fig_estoi = plot_metric_comparison(rps_metrics, baseline_metrics, 'estoi', 'ESTOI')
plt.show()

fig_pesq = plot_metric_comparison(rps_metrics, baseline_metrics, 'pesq', 'PESQ')
plt.show()

In [ ]:
# Combined bar plot
if rps_metrics is not None and baseline_metrics is not None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    metrics = ['si_sdr', 'estoi', 'pesq']
    titles = ['SI-SDR (dB)', 'ESTOI', 'PESQ']
    
    for ax, metric, title in zip(axes, metrics, titles):
        means = [rps_metrics[metric].mean(), baseline_metrics[metric].mean()]
        stds = [rps_metrics[metric].std(), baseline_metrics[metric].std()]
        
        x = np.arange(2)
        bars = ax.bar(x, means, yerr=stds, capsize=5, color=['#66c2a5', '#fc8d62'], alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(['DCUNet + RPS', 'Baseline'])
        ax.set_ylabel(title)
        ax.set_title(title)
        
        # Add value labels
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{mean:.2f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(RESULTS_RPS.parent / 'rps_experiment_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

## Sample Listening

Listen to and compare audio samples from both models.

In [ ]:
def get_sample_paths(sample_name, dataset_dir, rps_results, baseline_results):
    """Get all audio paths for a sample."""
    paths = {
        'mixture': dataset_dir / 'valid' / sample_name / 'mixture.wav',
        'clean': dataset_dir / 'valid' / sample_name / 'vocals.wav',
        'noise': dataset_dir / 'valid' / sample_name / 'noise.wav',
        'rps_enhanced': rps_results / 'eval_output' / sample_name / 'vocals.wav',
        'baseline_enhanced': baseline_results / 'eval_output' / sample_name / 'vocals.wav',
    }
    return {k: v for k, v in paths.items() if v.exists()}

def display_sample(sample_name):
    """Display audio players and spectrograms for a sample."""
    paths = get_sample_paths(sample_name, DATASET_DIR, RESULTS_RPS, RESULTS_BASELINE)
    
    if not paths:
        print(f"Sample {sample_name} not found")
        return
    
    print(f"Sample: {sample_name}")
    print("=" * 60)
    
    # Audio players
    for name, path in paths.items():
        audio, sr = sf.read(path)
        print(f"\n{name.replace('_', ' ').title()}:")
        display(ipd.Audio(audio, rate=sr))
    
    # Spectrograms
    n_plots = len(paths)
    fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3 * n_plots))
    if n_plots == 1:
        axes = [axes]
    
    for ax, (name, path) in zip(axes, paths.items()):
        audio, sr = sf.read(path)
        D = librosa.stft(audio, n_fft=2048, hop_length=512)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
        
        librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis='time', y_axis='hz', ax=ax)
        ax.set_title(name.replace('_', ' ').title())
    
    plt.tight_layout()
    plt.show()

In [ ]:
# List available samples
valid_samples = sorted([d.name for d in (DATASET_DIR / 'valid').iterdir() if d.is_dir()])
print(f"Available samples: {len(valid_samples)}")
print(valid_samples[:10])

In [ ]:
# Interactive sample selector
import ipywidgets as widgets
from ipywidgets import interact

@interact(sample=widgets.Dropdown(options=valid_samples, description='Sample:'))
def show_sample(sample):
    display_sample(sample)

## Per-SNR Analysis

Analyze performance across different input SNR levels.

In [ ]:
# Load dataset metadata to get SNR values
metadata_path = DATASET_DIR / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
    
    # Create SNR lookup
    snr_lookup = {}
    for split in ['train', 'valid']:
        if split in metadata:
            for item in metadata[split]:
                snr_lookup[item['id']] = item.get('input_snr', item.get('target_snr', np.nan))
    
    # Add SNR to metrics
    if rps_metrics is not None:
        rps_metrics['snr'] = rps_metrics['sample'].map(snr_lookup)
    if baseline_metrics is not None:
        baseline_metrics['snr'] = baseline_metrics['sample'].map(snr_lookup)
    
    print(f"Loaded SNR data for {len(snr_lookup)} samples")
else:
    print("Metadata file not found")

In [ ]:
# Plot SI-SDR vs input SNR
if rps_metrics is not None and 'snr' in rps_metrics.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.scatter(rps_metrics['snr'], rps_metrics['si_sdr'], alpha=0.6, label='DCUNet + RPS', s=30)
    if baseline_metrics is not None:
        ax.scatter(baseline_metrics['snr'], baseline_metrics['si_sdr'], alpha=0.6, label='Baseline', s=30)
    
    ax.set_xlabel('Input SNR (dB)')
    ax.set_ylabel('Output SI-SDR (dB)')
    ax.set_title('SI-SDR vs Input SNR')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add reference line (y = x would mean no improvement)
    xlim = ax.get_xlim()
    ax.plot(xlim, xlim, 'k--', alpha=0.3, label='No improvement')
    
    plt.tight_layout()
    plt.show()

## Statistical Significance Test

In [ ]:
from scipy import stats

if rps_metrics is not None and baseline_metrics is not None:
    # Paired t-test (samples should be matched)
    merged = rps_metrics.merge(baseline_metrics, on='sample', suffixes=('_rps', '_baseline'))
    
    print("Paired t-test results (RPS vs Baseline)")
    print("=" * 50)
    
    for metric in ['si_sdr', 'estoi', 'pesq']:
        rps_col = f'{metric}_rps'
        baseline_col = f'{metric}_baseline'
        
        if rps_col in merged.columns and baseline_col in merged.columns:
            rps_vals = merged[rps_col].dropna()
            baseline_vals = merged[baseline_col].dropna()
            
            # Ensure same samples
            valid_idx = rps_vals.index.intersection(baseline_vals.index)
            
            t_stat, p_value = stats.ttest_rel(
                merged.loc[valid_idx, rps_col],
                merged.loc[valid_idx, baseline_col]
            )
            
            diff_mean = (merged.loc[valid_idx, rps_col] - merged.loc[valid_idx, baseline_col]).mean()
            
            sig = "*" if p_value < 0.05 else ""
            sig += "*" if p_value < 0.01 else ""
            sig += "*" if p_value < 0.001 else ""
            
            print(f"{metric.upper():>8}: diff = {diff_mean:+.4f}, t = {t_stat:.3f}, p = {p_value:.4f} {sig}")

## Export Results

In [ ]:
# Save detailed results
output_dir = RESULTS_RPS.parent / 'rps_experiment_analysis'
output_dir.mkdir(exist_ok=True)

if rps_metrics is not None:
    rps_metrics.to_csv(output_dir / 'rps_metrics.csv', index=False)
if baseline_metrics is not None:
    baseline_metrics.to_csv(output_dir / 'baseline_metrics.csv', index=False)

print(f"Results saved to {output_dir}")